# Ejercicio 3 — Cambio de escala de una variable
**Universidad El Bosque — Programa de Matemáticas y Estadística — Introducción al Machine Learning — Semestre 2026-2**

Se ajusta el modelo $y=\beta_0+\beta_1x_1+\beta_2x_2$ y luego se repite el ajuste después de reescalar $z_2=x_2/100$, es decir, $y=\gamma_0+\gamma_1x_1+\gamma_2z_2$. Como $z_2$ es una función lineal de $x_2$, se espera que los dos modelos hagan exactamente las mismas predicciones, con $\hat\gamma_2=100\hat\beta_2$. El ejercicio lo verifica numéricamente.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sys

sys.path.append("../src")

# Formato numérico uniforme para las tablas
pd.set_option("display.float_format", lambda v: f"{v:.6f}")

import numpy as np
from lista4 import cargar_datos, resumen_regresion

df = cargar_datos("experimento_3")
df["z2"] = df["x2"] / 100          # variable reescalada
df.describe().loc[["mean", "std", "min", "max"]]

,x1,x2,y,z2
mean,0.031513,1.062102,0.110089,0.010621
std,1.047887,101.471545,7.405441,1.014715
min,-3.747414,-289.105312,-20.837760,-2.891053
max,3.150908,323.623238,21.269202,3.236232


La tabla muestra que `x2` tiene una escala de cientos, mientras que `z2` queda en una escala comparable a la de `x1`.

## (a) Coeficientes y riesgos de los dos modelos

In [2]:
m1 = resumen_regresion(df, ["x1", "x2"])    # modelo original  (beta)
m2 = resumen_regresion(df, ["x1", "z2"])    # modelo reescalado (gamma)

pd.DataFrame({
    "Modelo 1 (x1, x2)": {"Intercepto (beta0)": m1["intercepto"],
                          "coef x1 (beta1)":    m1["coeficientes"]["x1"],
                          "coef x2 (beta2)":    m1["coeficientes"]["x2"],
                          "Riesgo empírico":    m1["riesgo"]},
    "Modelo 2 (x1, z2)": {"Intercepto (beta0)": m2["intercepto"],
                          "coef x1 (beta1)":    m2["coeficientes"]["x1"],
                          "coef x2 (beta2)":    m2["coeficientes"]["z2"],
                          "Riesgo empírico":    m2["riesgo"]},
})

,"Modelo 1 (x1, x2)","Modelo 2 (x1, z2)"
Intercepto (beta0),-0.101470,-0.101470
coef x1 (beta1),5.012066,5.012066
coef x2 (beta2),0.050477,5.047693
Riesgo empírico,0.923225,0.923225


(En la columna del modelo 2, la fila `coef x2` corresponde en realidad al coeficiente de `z2`, es decir, a $\hat\gamma_2$.)

## (b) Diferencia máxima entre las predicciones

In [3]:
d_i = np.abs(m1["y_pred"] - m2["y_pred"])
max_d = d_i.max()
print(f"max_i d_i = {max_d:.3e}")

max_i d_i = 5.385e-15


La diferencia máxima entre las predicciones de los dos modelos es del orden de $10^{-15}$, es decir, error de redondeo de punto flotante. Para efectos prácticos, los dos modelos predicen exactamente lo mismo en todas las observaciones.

## (c) Verificación de $\hat\gamma_0\approx\hat\beta_0$, $\hat\gamma_1\approx\hat\beta_1$, $\hat\gamma_2\approx100\hat\beta_2$

In [4]:
beta0, beta1, beta2   = m1["intercepto"], m1["coeficientes"]["x1"], m1["coeficientes"]["x2"]
gamma0, gamma1, gamma2 = m2["intercepto"], m2["coeficientes"]["x1"], m2["coeficientes"]["z2"]

verificacion = pd.DataFrame({
    "Valor gamma":          [gamma0, gamma1, gamma2],
    "Valor esperado (beta)": [beta0, beta1, 100 * beta2],
}, index=["gamma0 vs beta0", "gamma1 vs beta1", "gamma2 vs 100·beta2"])
verificacion["Error absoluto"] = (verificacion["Valor gamma"] - verificacion["Valor esperado (beta)"]).abs()

# Los errores son muy pequeños, así que se muestran en notación científica
with pd.option_context("display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-3 else f"{v:.6f}"):
    display(verificacion)

,Valor gamma,Valor esperado (beta),Error absoluto
gamma0 vs beta0,-0.101470,-0.101470,2.776e-17
gamma1 vs beta1,5.012066,5.012066,8.882e-16
gamma2 vs 100·beta2,5.047693,5.047693,8.882e-16


Las tres relaciones se cumplen: los errores absolutos están en el orden del error de redondeo (entre $10^{-17}$ y $10^{-15}$). Teóricamente tiene sentido porque, al sustituir $z_2=x_2/100$, el modelo $\gamma_0+\gamma_1x_1+\gamma_2x_2/100$ coincide con el original solo si $\gamma_2/100=\beta_2$, y el mínimo de la pérdida cuadrática (que es único) se alcanza justamente ahí.

## (d) Comparación de $|\hat\beta_2|$ y $|\hat\gamma_2|$ y de las predicciones

In [5]:
print(f"|beta2|  = {abs(beta2):.6f}")
print(f"|gamma2| = {abs(gamma2):.6f}")
print(f"|gamma2| / |beta2| = {abs(gamma2) / abs(beta2):.4f}")
print(f"max_i d_i = {max_d:.3e}")

|beta2|  = 0.050477
|gamma2| = 5.047693
|gamma2| / |beta2| = 100.0000
max_i d_i = 5.385e-15


En valor absoluto, $|\hat\gamma_2|\approx 5.05$ es unas 100 veces mayor que $|\hat\beta_2|\approx 0.0505$, y a simple vista parecería que `z2` "importa" mucho más que `x2`. Sin embargo, las predicciones de los dos modelos son idénticas (diferencia máxima ≈ $10^{-15}$) y los riesgos coinciden. Moraleja: **la magnitud de un coeficiente depende de las unidades de la variable**, así que comparar coeficientes entre variables con escalas distintas no mide su importancia. Cambiar la escala de una variable solo cambia la parametrización del modelo, no el modelo ajustado ni su calidad.

## (e) Tabla resumen

In [6]:
tabla_3 = pd.DataFrame({
    "Intercepto":      [beta0, gamma0],
    "coef x1":         [beta1, gamma1],
    "coef x2 / z2":    [beta2, gamma2],
    "Riesgo empírico": [m1["riesgo"], m2["riesgo"]],
    "max_i d_i":       [max_d, max_d],
}, index=["Modelo 1: (x1, x2)", "Modelo 2: (x1, z2)"])

with pd.option_context("display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-3 else f"{v:.6f}"):
    display(tabla_3)

,Intercepto,coef x1,coef x2 / z2,Riesgo empírico,max_i d_i
"Modelo 1: (x1, x2)",-0.101470,5.012066,0.050477,0.923225,5.385e-15
"Modelo 2: (x1, z2)",-0.101470,5.012066,5.047693,0.923225,5.385e-15


El valor `max_i d_i` es una propiedad de la pareja de modelos, por eso aparece repetido en las dos filas. Los coeficientes de `x2` y `z2` difieren por un factor de 100, pero todo lo demás (intercepto, coeficiente de `x1`, riesgo y predicciones) coincide.